In [148]:
import time
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from pathlib import Path
import schwabdev as sd
import time as tm
import zarr
import shutil
import os

In [146]:
import warnings

warnings.filterwarnings('ignore', message="The data type .* does not have a Zarr V3 specification.*")
warnings.filterwarnings('ignore', message="Consolidated metadata is currently not part in the Zarr format 3 specification.*")

In [157]:
def gen_quote_data(day, time, symbols):
    data_bid = np.random.rand(len(symbols)) * 100
    data_ask = data_bid + (np.random.rand(len(symbols)) * 0.02)
    data_volume = np.random.rand(len(symbols)) * 1000
    return pd.DataFrame({'bid':data_bid,'ask':data_ask,'volume':data_volume}, index=symbols)

def gen_fundamental_data(day, symbols):
    data_pe_ratio = np.random.rand(len(symbols)) * 20
    data_market_cap = np.full()
    return pd.DataFrame({'pe_ratio':data_pe_ratio,'market_cap':data_market_cap}, index=symbols)

def create_db(day, time, symbols):
    db_path = 'test_data/hot.zarr'
    if os.path.exists(db_path):
        shutil.rmtree(db_path)
    # This function now just creates the shell for the first day
    add_day_shell(day, symbols, is_initial_creation=True)
    # And updates the first data point
    update_db_time(day, time, symbols)
    update_db_day_data(day, symbols)
    print(f"Initial database created for {day}.")


def add_day_shell(day, new_symbols, is_initial_creation=False):
    """
    Adds a new day shell. If the symbols have changed, it rebuilds the entire
    database with a combined list of symbols.
    """
    db_path = 'test_data/hot.zarr'
    temp_db_path = 'test_data/hot_temp.zarr'

    if is_initial_creation:
        existing_symbols = []
    else:
        ds_disk = xr.open_zarr(db_path, consolidated=True)
        existing_symbols = ds_disk.ident.values.tolist()

    old_set = set(existing_symbols)
    new_set = set(new_symbols)

    if old_set == new_set and not is_initial_creation:
        print("Symbol list unchanged. Appending new day shell.")
        final_symbols = existing_symbols
        ds_shell = _create_day_nan_dataset(day, final_symbols)
        ds_shell.to_zarr(db_path, mode='a-', append_dim='day')
        return

    print("Symbol list has changed or initial creation. Rebuilding database.")
    final_symbols = sorted(list(old_set.union(new_set)))
    
    if os.path.exists(temp_db_path):
        shutil.rmtree(temp_db_path)

    if not is_initial_creation:
        for existing_day in ds_disk.day.values:
            reindexed_shell = _create_day_nan_dataset(existing_day, final_symbols)
            
            # --- FIX IS HERE ---
            # Select with a list to preserve the 'day' dimension
            original_day_data = ds_disk.sel(day=[existing_day])
            
            reindexed_data = original_day_data.reindex({'ident': final_symbols})
            reindexed_shell.update(reindexed_data)

            mode = 'w' if not os.path.exists(temp_db_path) else 'a-'
            append_dim = None if mode == 'w' else 'day'
            reindexed_shell.to_zarr(temp_db_path, mode=mode, append_dim=append_dim, consolidated=True)
        ds_disk.close()

    new_day_shell = _create_day_nan_dataset(day, final_symbols)
    mode = 'w' if not os.path.exists(temp_db_path) else 'a-'
    append_dim = None if mode == 'w' else 'day'
    new_day_shell.to_zarr(temp_db_path, mode=mode, append_dim=append_dim, consolidated=True)

    if os.path.exists(db_path):
        shutil.rmtree(db_path)
    os.rename(temp_db_path, db_path)
    print(f"Database rebuild complete. Current symbols: {len(final_symbols)}")


def _create_day_nan_dataset(day, symbols):
    """Helper function to create a single-day dataset filled with NaNs."""
    time_coords = pd.date_range(start='00:00', end='23:55', freq='5min').strftime('%H:%M').to_list()
    
    nan_quote_data = np.full((1, len(time_coords), len(symbols), 3), np.nan)
    nan_fundamental_data = np.full((1, len(symbols), 2),             np.nan)
    
    coords = {
        'day': [day],
        'time': time_coords,
        'ident': symbols,
        'qVar': ['bid', 'ask', 'volume'],
        'fVar': ['pe_ratio', 'market_cap']
    }
    
    data = {
        '5m': (['day', 'time', 'ident', 'qVar'], nan_quote_data),
        '1d': (['day', 'ident', 'fVar'], nan_fundamental_data)
    }
    return xr.Dataset(data, coords=coords)

# --- Update functions remain the same ---
def update_db_time(day, time, symbols):
    """
    Updates a specific time slot for a given day with new data.
    """
    # 1. Generate the data we want to write
    test_data = gen_quote_data(day, time, symbols)
    
    ds_disk = xr.open_zarr('test_data/hot.zarr', consolidated=True)
    
    # 2. Find the location for the day and time slices
    day_idx = np.where(ds_disk.day.values == day)[0][0]
    time_idx = np.where(ds_disk.time.values == time)[0][0]
    
    # --- FIX IS HERE ---
    
    # 3. Create a full-sized NaN array that matches the shape of the target slice in Zarr
    existing_symbols = ds_disk.ident.values.tolist()
    full_slice_data = np.full((1, 1, len(existing_symbols), 3), np.nan)
    
    # 4. Find the integer positions where our new data should be placed
    target_indices = [existing_symbols.index(s) for s in symbols]
    
    # 5. Place the new data into the full_slice_data array at the correct positions
    full_slice_data[0, 0, target_indices, :] = test_data.to_numpy()
    
    # 6. Define a SIMPLE region that does NOT include the 'ident' dimension
    region_to_update = {
        'day': slice(day_idx, day_idx + 1), 
        'time': slice(time_idx, time_idx + 1)
    }

    # 7. Create a Dataset containing the complete, correctly ordered slice
    dataset_to_write = xr.Dataset({
        '5m': (['day', 'time', 'ident', 'qVar'], full_slice_data)
    })
    
    # 8. Write the full block to the simple region
    dataset_to_write.to_zarr(
        'test_data/hot.zarr', 
        mode='r+',
        region=region_to_update
    )
    ds_disk.close()
    print(f"Updated intraday data for day {day} at {time}.")

def update_db_day_data(day, symbols):
    """
    Updates the fundamental data for a specific day.
    """
    fundamental_data = gen_fundamental_data(day, symbols)
    ds_disk = xr.open_zarr('test_data/hot.zarr', consolidated=True)
    day_idx = np.where(ds_disk.day.values == day)[0][0]

    # --- APPLYING THE SAME FIX ---
    
    existing_symbols = ds_disk.ident.values.tolist()
    full_slice_data = np.full((1, len(existing_symbols), 2), np.nan)
    target_indices = [existing_symbols.index(s) for s in symbols]
    full_slice_data[0, target_indices, :] = fundamental_data.to_numpy()

    region_to_update = {
        'day': slice(day_idx, day_idx + 1)
    }
    
    dataset_to_write = xr.Dataset({
        '1d': (['day', 'ident', 'fVar'], full_slice_data)
    })
    
    dataset_to_write.to_zarr(
        'test_data/hot.zarr', 
        mode='r+',
        region=region_to_update
    )
    ds_disk.close()
    print(f"Updated fundamental data for day {day}.")


# -------------------------------------------------------------------
#  EXAMPLE USAGE
# -------------------------------------------------------------------

date_1 = '2025-08-25'
date_2 = '2025-08-26'
time_1 = '09:30'

# Initial symbols: 4 total
syms_1 = ['AAPL', 'NVDA', 'GOOG', 'F']
# New symbols for day 2: adds TSLA, removes GOOG. Total unique symbols will be 5.
syms_2 = ['AAPL', 'NVDA', 'F', 'TSLA']

# 1. Create the initial database with the first set of symbols
create_db(date_1, time_1, syms_1)

# 2. Add a new day with a DIFFERENT symbol list.
# This will trigger the re-indexing logic.
add_day_shell(date_2, syms_2)

# 3. Update some data for the new day
update_db_time(date_2, '10:00', syms_2)
update_db_day_data(date_2, syms_2)


# 4. Verification
print("\n--- Verification ---")
db = xr.open_zarr('test_data/hot.zarr', consolidated=True)

print(f"Final symbol list in database: {db.ident.values}")

print("\nData for Day 1 (2025-08-25):")
day_1_data = db['1d'].sel(day=date_1).to_pandas()
print(day_1_data)
print("Note: TSLA is present but all its values are NaN, as it was backfilled.")

print("\nData for Day 2 (2025-08-26):")
day_2_data = db['1d'].sel(day=date_2).to_pandas()
print(day_2_data)
print("Note: GOOG is present but its values are NaN, as it didn't have data for this day.")

Symbol list has changed or initial creation. Rebuilding database.
Database rebuild complete. Current symbols: 4
Updated intraday data for day 2025-08-25 at 09:30.
Updated fundamental data for day 2025-08-25.
Initial database created for 2025-08-25.
Symbol list has changed or initial creation. Rebuilding database.
Database rebuild complete. Current symbols: 5
Updated intraday data for day 2025-08-26 at 10:00.
Updated fundamental data for day 2025-08-26.

--- Verification ---
Final symbol list in database: ['AAPL' 'F' 'GOOG' 'NVDA' 'TSLA']

Data for Day 1 (2025-08-25):
fVar    pe_ratio    market_cap
ident                         
AAPL   18.495704  2.976439e+11
F       1.879786  4.967902e+11
GOOG    2.250058  9.154434e+11
NVDA   10.080180  7.894220e+11
TSLA         NaN           NaN
Note: TSLA is present but all its values are NaN, as it was backfilled.

Data for Day 2 (2025-08-26):
fVar    pe_ratio    market_cap
ident                         
AAPL    4.963939  2.801353e+11
F       9.5243

In [130]:
def gen_quote_data(day,time,symbols):
    data_bid = np.random.rand(len(symbols)) * 100
    data_ask = data_bid + (np.random.rand(len(symbols)) * 0.02)
    data_volume = np.random.rand(len(symbols)) * 1000
    return pd.DataFrame({'bid':data_bid,'ask':data_ask,'volume':data_volume},index=symbols)

def gen_funamental_data(day,symbols):
    data_pe_ratio = np.random.rand(len(symbols)) * 20
    data_market_cap = np.random.rand(len(symbols)) * 1e12
    return pd.DataFrame({'pe_ratio':data_pe_ratio,'market_cap':data_market_cap},index=symbols)

def gen_fundamental_nan(day,symbols):
    data_pe_ratio = np.full(len(symbols), np.nan)
    data_market_cap = np.full(len(symbols), np.nan)
    return pd.DataFrame({'pe_ratio':data_pe_ratio,'market_cap':data_market_cap},index=symbols)

In [143]:
def create_db(day,time,symbols):
    test_data = gen_quote_data(day,time,symbols)
    test_data_f = gen_funamental_data(day,symbols)
    
    coords = {
        'day':[day],
        'time':[time],
        'ident':symbols,
        'qVar': ['bid','ask','volume'],
        'fVar': ['pe_ratio','market_cap']
    }
    
    data = {}
    data['5m'] = (['day','time','ident','qVar'], test_data.to_numpy()[np.newaxis,np.newaxis,:,:])
    data['1d'] = (['day','ident','fVar'], test_data_f.to_numpy()[np.newaxis,:,:])

    ds = xr.Dataset(data,coords=coords)

    ds.to_zarr('test_data/hot.zarr',mode='w',consolidated=True)

def append_db_time(day,time,symbols):
    test_data = gen_quote_data(day,time,symbols)
    coords = {
        'day':[day],
        'time':[time],
        'ident':symbols,
        'qVar': ['bid','ask','volume'],
        'fVar': ['pe_ratio','market_cap']
    }
    to_add = {}
    to_add['5m'] = (['day','time','ident','qVar'], test_data.to_numpy()[np.newaxis,np.newaxis,:,:])
    after = xr.Dataset(to_add,coords=coords)
    after.to_zarr('test_data/hot.zarr',mode='a-',append_dim='time')

def append_db_day(day,time,symbols):
    test_data = gen_quote_data(day,time,symbols)
    test_data_f = gen_fundamental_nan(day,symbols)

    coords = {
        'day':[day],
        'time':[time],
        'ident':symbols,
        'qVar': ['bid','ask','volume'],
        'fVar': ['pe_ratio','market_cap']
    }

    data = {}
    data['5m'] = (['day','time','ident','qVar'], test_data.to_numpy()[np.newaxis,np.newaxis,:,:])
    data['1d'] = (['day','ident','fVar'], test_data_f.to_numpy()[np.newaxis,:,:])

    ds = xr.Dataset(data,coords=coords)

    ds.to_zarr('test_data/hot.zarr',mode='a',append_dim='day')

def append_db_day_new_symbols(day,time,symbols):
    pass

def update_db_day_data(day,time,symbols):
    pass


In [137]:
date_1 = '2025-08-25'
date_2 = '2025-08-26'
time_1 = '14:35'
time_2 = '14:40'
time_3 = '14:50'
syms_1 = ['AAPL','NVDA','GOOG','F']
syms_2 = ['AAPL','NVDA','GOOG','F','TSLA']
syms_3 = ['AAPL','NVDA','F','TSLA']

In [144]:
create_db(date_1,time_1,syms_1)
xr.open_dataset('test_data/hot.zarr')

<xarray.Dataset> Size: 436B
Dimensions:  (day: 1, ident: 4, fVar: 2, time: 1, qVar: 3)
Coordinates:
  * fVar     (fVar) <U10 80B 'pe_ratio' 'market_cap'
  * day      (day) <U10 40B '2025-08-25'
  * ident    (ident) <U4 64B 'AAPL' 'NVDA' 'GOOG' 'F'
  * qVar     (qVar) <U6 72B 'bid' 'ask' 'volume'
  * time     (time) <U5 20B '14:35'
Data variables:
    1d       (day, ident, fVar) float64 64B ...
    5m       (day, time, ident, qVar) float64 96B ...

In [145]:
append_db_time(date_1,time_2,syms_1)
append_db_time(date_1,time_3,syms_1)

append_db_day(date_2,time_1,syms_1)

ValueError: variable '5m' already exists with different dimension sizes: {'time': 3, 'ident': 4, 'qVar': 3} != {'time': 1, 'ident': 4, 'qVar': 3}. to_zarr() only supports changing dimension sizes when explicitly appending, but append_dim='day'. If you are attempting to write to a subset of the existing store without changing dimension sizes, consider using the region argument in to_zarr().

In [135]:
# ds['5m'].sel(day='2025-08-25')
# ds['5m'].sel(time='17:45',day='2025-08-25',qVar='ask').mean()
xr.open_dataset('test_data/hot.zarr')

<xarray.Dataset> Size: 668B
Dimensions:  (day: 1, time: 3, ident: 4, qVar: 3, fVar: 2)
Coordinates:
  * ident    (ident) <U4 64B 'AAPL' 'NVDA' 'GOOG' 'F'
  * qVar     (qVar) <U6 72B 'bid' 'ask' 'volume'
  * day      (day) <U10 40B '2025-08-25'
  * time     (time) <U5 60B '14:35' '14:40' '14:50'
  * fVar     (fVar) <U10 80B 'pe_ratio' 'market_cap'
Data variables:
    5m       (day, time, ident, qVar) float64 288B ...
    1d       (day, ident, fVar) float64 64B ...